# Downstream performance versus feature-extraction FLOPs

This notebook recreates the structure of CSMoE Figures 7 and 8 and adds this model under both downstream spatial protocols. The main `224 x 224` point follows the CSMoE input convention. The `64 x 64` point is an explicitly labeled native-resolution operating point, not a strict protocol match.

FLOPs use the CSMoE/TerraTorch convention: one encoder feature-extraction pass, batch size one, Sentinel-2 only, no MAE decoder, metadata, or downstream head, and one multiply-add counted as one operation. The counter was validated by reproducing Satlas at `17.1201G`, matching CSMoE Table I (`17.12G`). Our validated costs are `51.4767G` at 224 and `1.0290G` at 64 with all 13 supported S2 bands.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd


def find_repo_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'models' / 'moe_mae.py').exists():
            return candidate
    raise FileNotFoundError('Run the notebook from inside the repository.')


repo_root = find_repo_root()
experiment_candidates = [
    Path(os.environ['MEOX_EXPERIMENT_ROOT'])
    if os.environ.get('MEOX_EXPERIMENT_ROOT') else None,
]
experiment_root = next(
    (path for path in experiment_candidates if path is not None and path.exists()), None
)
if experiment_root is None:
    raise FileNotFoundError(
        'Set GEO_MOE_EXPERIMENT_ROOT to the directory containing the eval folder.'
    )

eval_root = experiment_root / 'eval'
figure_dir = eval_root / 'efficiency_figures'
figure_dir.mkdir(parents=True, exist_ok=True)
reference_path = repo_root / 'benchmarks' / 'csmoe_efficiency_reference.csv'

print('Experiment:', experiment_root)
print('Figures:', figure_dir)

## Reference-data provenance

CSMoE variant values come directly from Table IV. Cashew values explicitly stated in Section IV-B are also recorded as exact. Remaining comparator metrics were recovered from the vector marker coordinates in the official arXiv Figure 7/8 PDFs and rounded to the displayed one-decimal precision. FLOPs and parameter counts come from CSMoE Table I. The `source_type` column keeps exact and digitized values distinguishable.

In [ ]:
references = pd.read_csv(reference_path).fillna({'variant': ''})

dataset_specs = {
    'm-bigearthnet': ('classification', 'micro_mAP', 'micro_map'),
    'm-brick-kiln': ('classification', 'average_accuracy', 'average_accuracy'),
    'm-eurosat': ('classification', 'average_accuracy', 'average_accuracy'),
    'm-so2sat': ('classification', 'average_accuracy', 'average_accuracy'),
    'm-cashew-plant': ('segmentation', 'mean_iou', 'mean_iou'),
    'm-SA-crop-type': ('segmentation', 'mean_iou', 'mean_iou'),
}
protocols = {
    'geobench_224': {
        'label': 'Ours 224', 'image_size': 224, 'flops_g': 51.476678208,
        'classification_dir': 'geobench_classification_224',
        'segmentation_dir': 'geobench_segmentation_224',
    },
    'model_input': {
        'label': 'Ours 64', 'image_size': 64, 'flops_g': 1.029031488,
        'classification_dir': 'geobench_classification_model_input',
        'segmentation_dir': 'geobench_segmentation_model_input',
    },
}
our_parameters_m = 3.114993

our_rows = []
missing_rows = []
for dataset, (task, metric, result_key) in dataset_specs.items():
    for protocol, protocol_spec in protocols.items():
        folder = protocol_spec[f'{task}_dir']
        result_path = eval_root / folder / dataset / 'results.json'
        if not result_path.exists():
            missing_rows.append({'dataset': dataset, 'protocol': protocol, 'status': 'pending'})
            continue
        with result_path.open() as handle:
            result = json.load(handle)
        if task == 'classification':
            aggregate = result['linear_probe']['aggregate'][result_key]
        else:
            aggregate = result['segmentation_probe']['aggregate_test'][result_key]
        our_rows.append({
            'dataset': dataset, 'task': task, 'metric': metric,
            'model': protocol_spec['label'], 'protocol': protocol,
            'image_size': protocol_spec['image_size'],
            'flops_g': protocol_spec['flops_g'],
            'parameters_m': our_parameters_m,
            'value': 100 * aggregate['mean'],
            'std': 100 * aggregate['std'],
            'source': str(result_path),
        })

ours = pd.DataFrame(our_rows)
missing = pd.DataFrame(missing_rows)
display(ours.sort_values(['task', 'dataset', 'image_size']).round(4))
if not missing.empty:
    print('Pending result files:')
    display(missing)

## Reading the plots

The x-axis is logarithmic. Literature circles use model-specific colors and their area is scaled from parameter count with a small minimum size for visibility. CSMoE variants are connected and labeled by patch size. Our points use stars and five-seed error bars. The dashed segment between our 64 and 224 points shows only our resolution/computation trade-off; it must not be interpreted as two equally matched CSMoE-protocol evaluations.

In [ ]:
model_colors = {
    'CSMAE': '#4C78A8', 'DOFA': '#F58518',
    'Prithvi V2-300': '#54A24B', 'Prithvi V2-600': '#8E6C8A',
    'Satlas': '#9D755D', 'TerraMind': '#E17C9A', 'CSMoE': '#D62728',
}
dataset_titles = {
    'm-bigearthnet': 'BigEarthNet', 'm-brick-kiln': 'Brick Kiln',
    'm-eurosat': 'EuroSAT', 'm-so2sat': 'So2Sat',
    'm-cashew-plant': 'Cashew Plant', 'm-SA-crop-type': 'SA Crop Type',
}
metric_labels = {
    'micro_mAP': 'Micro mAP [%]', 'average_accuracy': 'Average accuracy [%]',
    'mean_iou': 'Mean IoU [%]',
}


def bubble_area(parameters_m):
    return 36 + 0.55 * parameters_m


def plot_dataset(axis, dataset):
    reference = references[references['dataset'] == dataset]
    baselines = reference[reference['model'] != 'CSMoE']
    for _, row in baselines.iterrows():
        axis.scatter(
            row.flops_g, row.value, s=bubble_area(row.parameters_m),
            color=model_colors[row.model], edgecolor='white', linewidth=0.7,
            alpha=0.9, zorder=2,
        )

    csmoe = reference[reference['model'] == 'CSMoE'].sort_values('flops_g')
    axis.plot(csmoe.flops_g, csmoe.value, color=model_colors['CSMoE'], alpha=0.45)
    axis.scatter(
        csmoe.flops_g, csmoe.value, s=csmoe.parameters_m.map(bubble_area),
        color=model_colors['CSMoE'], edgecolor='white', linewidth=0.8, zorder=3,
    )
    for index, (_, row) in enumerate(csmoe.iterrows()):
        patch_size = row.variant.replace('patch', '')
        axis.annotate(
            f'ρ={patch_size}', (row.flops_g, row.value),
            xytext=(3, -13 if index % 2 == 0 else 7),
            textcoords='offset points', fontsize=7, color='#9B1C1C',
        )

    our_data = ours[ours['dataset'] == dataset].sort_values('flops_g')
    if len(our_data) == 2:
        axis.plot(
            our_data.flops_g, our_data.value, '--', color='#1A1A1A',
            linewidth=1.2, alpha=0.65, zorder=4,
        )
    for _, row in our_data.iterrows():
        is_native = row.protocol == 'model_input'
        axis.errorbar(
            row.flops_g, row.value, yerr=row['std'], fmt='none',
            ecolor='#111111', elinewidth=1.2, capsize=3, zorder=5,
        )
        axis.scatter(
            row.flops_g, row.value, s=max(90, bubble_area(row.parameters_m)),
            marker='*', facecolor='white' if is_native else '#111111',
            edgecolor='#111111', linewidth=1.2, zorder=6,
        )
        axis.annotate(
            row.model, (row.flops_g, row.value), xytext=(5, 6),
            textcoords='offset points', fontsize=8, weight='bold',
        )

    metric = reference.iloc[0].metric
    axis.set_title(dataset_titles[dataset], loc='left', weight='bold')
    axis.set_ylabel(metric_labels[metric])
    axis.set_xlabel('Feature-extraction FLOPs [G]')
    axis.set_xscale('log')
    axis.set_xlim(0.75, 220)
    axis.grid(True, which='major', color='#D8D4CB', linewidth=0.8)
    axis.grid(True, which='minor', axis='x', color='#ECE8DF', linewidth=0.5)
    axis.set_facecolor('#FAF8F2')


legend_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=color,
           markeredgecolor='white', markersize=8, label=model)
    for model, color in model_colors.items()
] + [
    Line2D([0], [0], marker='*', color='none', markerfacecolor='#111111',
           markeredgecolor='#111111', markersize=11, label='Ours 224'),
    Line2D([0], [0], marker='*', color='none', markerfacecolor='white',
           markeredgecolor='#111111', markersize=11, label='Ours 64'),
]
parameter_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor='#888888',
           markeredgecolor='white', markersize=bubble_area(value) ** 0.5,
           label=f'{value:g}M')
    for value in (3, 100, 300, 600)
]

In [ ]:
classification_datasets = [
    'm-bigearthnet', 'm-brick-kiln', 'm-eurosat', 'm-so2sat'
]
figure, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
for axis, dataset in zip(axes.flat, classification_datasets):
    plot_dataset(axis, dataset)
figure.legend(
    handles=legend_handles, loc='outside lower center', ncol=5, frameon=False
)
axes[1, 1].legend(
    handles=parameter_handles, title='# parameters', loc='lower right',
    frameon=True, framealpha=0.9, ncol=2,
)
figure.suptitle('Frozen classification probes: performance versus FLOPs', weight='bold')
classification_path = figure_dir / 'classification_performance_vs_flops.pdf'
figure.savefig(classification_path, bbox_inches='tight')
figure.savefig(classification_path.with_suffix('.png'), dpi=300, bbox_inches='tight')
plt.show()
print(classification_path)

In [ ]:
segmentation_datasets = ['m-cashew-plant', 'm-SA-crop-type']
figure, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for axis, dataset in zip(axes, segmentation_datasets):
    plot_dataset(axis, dataset)
    if ours[ours['dataset'] == dataset].empty:
        axis.text(
            0.98, 0.04, 'Our result pending', transform=axis.transAxes,
            ha='right', va='bottom', fontsize=9, color='#555555',
        )
figure.legend(
    handles=legend_handles, loc='outside lower center', ncol=5, frameon=False
)
axes[1].legend(
    handles=parameter_handles, title='# parameters', loc='upper left',
    frameon=True, framealpha=0.9, ncol=2,
)
figure.suptitle('Frozen segmentation probes: performance versus FLOPs', weight='bold')
segmentation_path = figure_dir / 'segmentation_performance_vs_flops.pdf'
figure.savefig(segmentation_path, bbox_inches='tight')
figure.savefig(segmentation_path.with_suffix('.png'), dpi=300, bbox_inches='tight')
plt.show()
print(segmentation_path)

## Reporting rules

Use `Ours 224` for direct claims against CSMoE because both metrics and FLOPs use the paper's 224-pixel convention. Use `Ours 64` only to discuss the native-resolution efficiency/performance trade-off. Preserve the distinction between exact table/text values and vector-digitized comparator values in captions or supplementary material. Regenerate both figures after every missing `results.json` has been added; no notebook code change is required.